## Implementação da camada Bronze — `ecommerce_produtos`

Este notebook lê arquivos `.parquet` do diretório `real-time-data` no container `raw`, adiciona apenas auditoria da Bronze e salva como **Delta físico** no diretório `bronze/ecommerce_produtos` do container `squad1`.

### O que foi adaptado aqui:

1. Troquei `saveAsTable` pelo mesmo mecanismo de gravação física via SDK `deltalake` (`write_deltalake`).
2. **Trouxe as funções para dentro deste notebook** (`get_delta_path`, `delta_existe`, `gravar_delta` e uma função extra para checar arquivos já processados).
3. Caso a pasta física esteja vazia, **todos os arquivos da RAW serão reprocessados nesta primeira execução**.
4. `PARTICIONAR_BRONZE = True`: o Delta é gravado com **particionamento físico Hive-style** pelas colunas `bronze_ingest_year`, `bronze_ingest_month`, `bronze_ingest_day` e `bronze_ingest_hour`. Isso significa que, fisicamente, a pasta `bronze/ecommerce_produtos` terá subpastas no formato `bronze_ingest_year=AAAA/bronze_ingest_month=MM/bronze_ingest_day=DD/bronze_ingest_hour=HH/`, cada uma contendo seus arquivos `part-*.parquet`, além do `_delta_log` na raiz da tabela. A função `gravar_delta` já decide as colunas de partição dinamicamente: ela só particiona quando `camada == "bronze"` e quando essas colunas existem no DataFrame — o que é exatamente o nosso caso.

## 1. Instalação do SDK `deltalake`

In [0]:
%pip install -q deltalake pyarrow

## 2. Imports, parâmetros e credenciais

In [0]:
# Imports e parâmetros

import os
import uuid
import pandas as pd
import pyarrow as pa
from io import BytesIO
from functools import reduce
from datetime import datetime, timezone
from dotenv import load_dotenv

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

from deltalake import DeltaTable
from deltalake.writer import write_deltalake

from pyspark.sql import functions as F

# --- Origem (RAW) ---
CONTAINER_RAW = "raw"
REAL_TIME_DATA = "real-time-data"
NOME_ARQUIVO = "ecommerce_produtos.parquet"

# --- Destino físico (Bronze, Delta) ---
CONTAINER_SQUAD1 = "squad1"
CAMADA_DESTINO = "bronze"
ENTIDADE_DESTINO = "ecommerce_produtos"
PASTA_DESTINO = f"{CAMADA_DESTINO}/{ENTIDADE_DESTINO}"

# Particionamento físico Hive-style habilitado: bronze_ingest_year/month/day/hour.
# A função gravar_delta() usa essas colunas como partition_by ao escrever no
# caminho físico (squad1/bronze/ecommerce_produtos/bronze_ingest_year=AAAA/...).
PARTICIONAR_BRONZE = True

RUN_ID = str(uuid.uuid4())
DATA_EXECUCAO = datetime.now(timezone.utc)

print("Container RAW:", CONTAINER_RAW)
print("Pasta real-time-data:", REAL_TIME_DATA)
print("Arquivo alvo:", NOME_ARQUIVO)
print("Destino físico Bronze:", f"{CONTAINER_SQUAD1}/{PASTA_DESTINO}")
print("RUN_ID:", RUN_ID)

# Definir as credenciais do Service Principal (.env):
load_dotenv("/Workspace/Users/soaress.elias@gmail.com/merca-data-platform-categorias/.env")

CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT_NAME = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

# Opções de storage no formato esperado pelo SDK deltalake (object_store/Azure)
STORAGE_OPTIONS = {
    "account_name": STORAGE_ACCOUNT_NAME,
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "tenant_id": TENANT_ID,
}

# Criar a credencial:
credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

# Criar o DataLakeServiceClient:
service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    credential=credential
)

## 3. Funções auxiliares da camada Bronze (integradas — sem `%run` externo)

In [0]:
def get_delta_path(camada: str, tabela: str, storage_opts: dict) -> str:
    """Monta a URI nativa abfss:// para a tabela Delta, tratando gravação na raiz (camada vazia)."""
    conta = storage_opts.get("account_name")
    if not camada:
        return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{tabela}"
    return f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/{camada}/{tabela}"


def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    """Verifica se já existe uma tabela Delta válida no caminho físico."""
    try:
        DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=storage_opts)
        return True
    except Exception:
        return False


def gravar_delta(df, camada: str, tabela: str, storage_opts: dict, mode: str = "append", particionar: bool = True) -> bool:
    """Grava um DataFrame Spark como Delta físico via SDK deltalake (bypass do Serverless)."""
    path = get_delta_path(camada, tabela, storage_opts)
    modo_real = mode if delta_existe(camada, tabela, storage_opts) else "overwrite"

    try:
        # 1. Conversão para Pandas
        pdf = df.toPandas()

        # 2. Correção OBRIGATÓRIA de fuso horário (evita erro fatal no PyArrow)
        for col_name in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col_name]):
                pdf[col_name] = pdf[col_name].dt.tz_localize(None)

        # 3. Conversão para Tabela Arrow
        tabela_arrow = pa.Table.from_pandas(pdf, preserve_index=False)

        # 4. Definição dinâmica de partições (se solicitado)
        partition_by = None
        if particionar and camada == "bronze":
            possiveis = ["bronze_ingest_year", "bronze_ingest_month", "bronze_ingest_day", "bronze_ingest_hour"]
            colunas_pdf = pdf.columns.tolist()
            partition_by = [c for c in possiveis if c in colunas_pdf] or None

        # 5. Gravação via SDK (bypass do Databricks Serverless)
        write_deltalake(
            table_or_uri=path,
            data=tabela_arrow,
            mode=modo_real,
            storage_options=storage_opts,
            partition_by=partition_by,
            schema_mode="overwrite" if modo_real == "overwrite" else "merge"
        )

        print(f"[Sucesso] Gravado fisicamente em: {path} | Linhas: {len(pdf)}")
        return True

    except Exception as e:
        print(f"[Erro] Falha ao gravar {path}: {str(e)}")
        return False


def obter_arquivos_ja_processados(camada: str, tabela: str, storage_opts: dict) -> set:
    """Lê a tabela Delta física (se existir) e retorna o conjunto de bronze_source_file já gravados."""
    if not delta_existe(camada, tabela, storage_opts):
        return set()
    try:
        dt = DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=storage_opts)
        pdf = dt.to_pandas(columns=["bronze_source_file"])
        return set(pdf["bronze_source_file"].dropna().unique().tolist())
    except Exception as e:
        print(f"[Aviso] Não foi possível ler arquivos já processados: {e}")
        return set()

## 4. Clients do ADLS para os containers `raw` e `squad1`

In [0]:
file_system_raw = service_client.get_file_system_client(file_system=CONTAINER_RAW)
file_system_squad1 = service_client.get_file_system_client(file_system=CONTAINER_SQUAD1)

print("File system client criado para o container:", CONTAINER_RAW)
print("File system client criado para o container:", CONTAINER_SQUAD1)

## 5. Localizar arquivos `.parquet` na RAW

In [0]:
arquivos_encontrados = [
    p.name
    for p in file_system_raw.get_paths(path=REAL_TIME_DATA, recursive=True)
    if (not p.is_directory) and p.name.endswith(NOME_ARQUIVO)
]

print(f"Arquivos encontrados para {NOME_ARQUIVO}: {len(arquivos_encontrados)}")
for arquivo in arquivos_encontrados:
    print(arquivo)

if len(arquivos_encontrados) == 0:
    raise Exception(f"Nenhum arquivo {NOME_ARQUIVO} encontrado em {CONTAINER_RAW}/{REAL_TIME_DATA}")

## 6. Identificar arquivos ainda não processados na Bronze física

Caso a tabela Delta física ainda não exista em `squad1/bronze/ecommerce_produtos`, todos os arquivos encontrados serão tratados como novos nesta primeira execução — o que é esperado.

In [0]:
arquivos_processados_set = obter_arquivos_ja_processados(
    camada=CAMADA_DESTINO,
    tabela=ENTIDADE_DESTINO,
    storage_opts=STORAGE_OPTIONS
)

arquivos_novos = [
    arquivo for arquivo in arquivos_encontrados
    if arquivo not in arquivos_processados_set
]

print(f"Arquivos já processados na Bronze física: {len(arquivos_processados_set)}")
print(f"Arquivos novos para processar: {len(arquivos_novos)}")

for arquivo in arquivos_novos:
    print("NOVO:", arquivo)

## 7. Ler arquivos novos e adicionar auditoria Bronze

In [0]:
dfs_spark = []

for arquivo in arquivos_novos:
    print(f"Lendo arquivo novo: {arquivo}")

    file_client = file_system_raw.get_file_client(arquivo)
    conteudo = file_client.download_file().readall()

    # Leitura do parquet via SDK (restrição de ABFSS direto no Serverless).
    # Em seguida convertemos para Spark DataFrame apenas para compor o micro-lote.
    df_pandas = pd.read_parquet(BytesIO(conteudo))
    df_spark = spark.createDataFrame(df_pandas)

    df_spark = (
        df_spark
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_source_file", F.lit(arquivo))
        .withColumn("bronze_run_id", F.lit(RUN_ID))
        .withColumn("bronze_ingest_year", F.year(F.col("bronze_ingested_at")))
        .withColumn("bronze_ingest_month", F.month(F.col("bronze_ingested_at")))
        .withColumn("bronze_ingest_day", F.dayofmonth(F.col("bronze_ingested_at")))
        .withColumn("bronze_ingest_hour", F.hour(F.col("bronze_ingested_at")))
    )

    dfs_spark.append(df_spark)

if len(dfs_spark) == 0:
    print("Nenhum arquivo novo para processar. A Bronze não será alterada.")
    df_bronze_micro_lote = None
else:
    df_bronze_micro_lote = reduce(
        lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
        dfs_spark
    )

    print("Registros no micro-lote Bronze:", df_bronze_micro_lote.count())
    display(df_bronze_micro_lote.limit(10))

## 8. Gravar Bronze como Delta físico (via SDK `deltalake`)

In [0]:
if df_bronze_micro_lote is not None:
    sucesso = gravar_delta(
        df=df_bronze_micro_lote,
        camada=CAMADA_DESTINO,
        tabela=ENTIDADE_DESTINO,
        storage_opts=STORAGE_OPTIONS,
        mode="append",
        particionar=PARTICIONAR_BRONZE
    )

    if sucesso:
        print("Carga Bronze gravada com sucesso (Delta físico via deltalake SDK)!")
    else:
        print("Falha ao gravar a carga Bronze. Veja a mensagem de erro acima.")
else:
    print("Carga Bronze ignorada: não havia arquivos novos.")

## 9. Validação física do resultado

In [0]:
print("Arquivos físicos dentro de squad1/bronze/ecommerce_produtos:")
try:
    for item in file_system_squad1.get_paths(path=PASTA_DESTINO, recursive=True):
        tipo = "\U0001F4C1" if item.is_directory else "\U0001F4C4"
        print(f"{tipo} {item.name}")
except Exception as e:
    print("Não foi possível listar o destino Bronze:", e)

print("\nValidação Delta:")
if delta_existe(camada=CAMADA_DESTINO, tabela=ENTIDADE_DESTINO, storage_opts=STORAGE_OPTIONS):
    print("Sucesso! Tabela Delta física validada em squad1/bronze/ecommerce_produtos.")

    dt = DeltaTable(get_delta_path(CAMADA_DESTINO, ENTIDADE_DESTINO, STORAGE_OPTIONS), storage_options=STORAGE_OPTIONS)
    pdf_validacao = dt.to_pandas()

    print("Total de registros:", len(pdf_validacao))
    print("Total de arquivos de origem distintos:", pdf_validacao["bronze_source_file"].nunique())

    print("\nPartições físicas (Hive-style) detectadas pelo deltalake:")
    particoes = dt.partitions()
    if particoes:
        for p in sorted(particoes, key=lambda d: (d.get("bronze_ingest_year"), d.get("bronze_ingest_month"), d.get("bronze_ingest_day"), d.get("bronze_ingest_hour"))):
            print(
                f"bronze_ingest_year={p.get('bronze_ingest_year')}/"
                f"bronze_ingest_month={p.get('bronze_ingest_month')}/"
                f"bronze_ingest_day={p.get('bronze_ingest_day')}/"
                f"bronze_ingest_hour={p.get('bronze_ingest_hour')}"
            )
        print(f"\nTotal de partições físicas: {len(particoes)}")
    else:
        print("Nenhuma partição encontrada (tabela sem particionamento físico).")
else:
    print("Atenção: a tabela Delta ainda não foi encontrada no caminho físico esperado.")